1. Загрузить текстовые корпуса
   - Корпус 1: Классическая литература, например глава из Братьев Карамазовых Ф.М. Достоевского
   - Корпус 2: Научная статья на 10 стр на русском языке.
2. Разбить корпуса на предложения
3. По каждому предложению сделать перевод на английский, а затем обратно на русский
   - При помощи deepseek (или иной гибер LLM)
   - При помощи Google Translate (или иного переводчика отличного от первого)
4. Посчитать средние метрики BLEU и METEOR между исходным предложением и дважды переведенным
5. Сделать мини сводную таблицу и вывод о качестве 2-х переводчиков

In [46]:
import re
from razdel import sentenize

def split_into_sentences(text):
    text = re.sub(r'\s+', ' ', text).strip()
    sentences = [sent.text for sent in sentenize(text)]
    return sentences

In [47]:
with open('dostoevsky.txt', 'r', encoding='utf-8') as f:
    dostoevsky_text = f.read()

with open('face_control_paper.txt', 'r', encoding='utf-8') as f:
    scientific_text = f.read()

with open('dostoevsky_deepseek_ru.txt', 'r', encoding='utf-8') as f:
    dostoevsky_deepseek_ru = f.read()

with open('face_control_deepseek_ru.txt', 'r', encoding='utf-8') as f:
    scientific_deepseek_ru = f.read()

In [48]:
dostoevsky_sentences = split_into_sentences(dostoevsky_text)
scientific_sentences = split_into_sentences(scientific_text)
dostoevsky_deepseek_sentences = split_into_sentences(dostoevsky_deepseek_ru)
scientific_deepseek_sentences = split_into_sentences(scientific_deepseek_ru)

In [49]:
print(f"Number of sentences: {len(dostoevsky_sentences)}")
print("Example:", dostoevsky_sentences[0])

Number of sentences: 95
Example: Приступая к описанию недавних и столь странных событий, происшедших в нашем, доселе ничем не отличавшемся городе, я принужден, по неумению моему, начать несколько издалека, а именно некоторыми биографическими подробностями о талантливом и многочтимом Степане Трофимовиче Верховенском.


In [50]:
scientific_sentences = split_into_sentences(scientific_text)
print(f"Number of sentences: {len(scientific_sentences)}")
print("Example:", scientific_sentences[0])

Number of sentences: 136
Example: ТЕХНОЛОГИЯ ВЕРИФИКАЦИИ ЛИЧНОСТИ ЧЕЛОВЕКА ДЛЯ МОБИЛЬНЫХ УСТРОЙСТВ ПО УДОСТОВЕРЯЮЩЕМУ ДОКУМЕНТУ И СНИМКУ ЛИЦА НА ОДНОМ ИЗОБРАЖЕНИИ Верификация личности клиента по его автопортрету и фотографии на удостоверяющем документе активно используется в аэропортах, банках и госучреждениях для ускорения обслуживания и снижения влияния человеческого фактора.


In [51]:
# import os
# import requests
# from dotenv import load_dotenv

# load_dotenv()
# DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
# DEEPSEEK_API_URL = "https://api.deepseek.com/chat/completions"

# def translate_with_deepseek(text, source_lang="ru", target_lang="en"):
#     if not DEEPSEEK_API_KEY:
#         raise ValueError("API-key for DeepSeek not found")

#     headers = {
#         "Authorization": f"Bearer {DEEPSEEK_API_KEY}",
#         "Content-Type": "application/json"
#     }

#     prompt = (
#         f"Translate the following text from {source_lang} to {target_lang}. "
#         f"Output ONLY the translation, without any explanations, quotes, or notes.\n\n"
#         f"{text}"
#     )

#     data = {
#         "model": "deepseek-chat",
#         "messages": [{"role": "user", "content": prompt}],
#         "temperature": 0.0
#     }

#     try:
#         response = requests.post(DEEPSEEK_API_URL, headers=headers, json=data, timeout=30)
#         if response.status_code != 200:
#             print(f"API error: {response.status_code}, {response.text}")
#         response.raise_for_status()
#     except requests.exceptions.RequestException as e:
#         print(f"Request exception: {e}")
#         return text  

#     translated_text = response.json()["choices"][0]["message"]["content"].strip()
#     return translated_text

In [52]:
from deep_translator import GoogleTranslator

# def translate_with_google(text, source_lang="ru", target_lang="en"):
#     translator = GoogleTranslator(source=source_lang, target=target_lang)
#     return translator.translate(text)

In [53]:
def get_deepseek_translation(index, deepseek_sentences):
    if index < len(deepseek_sentences):
        return deepseek_sentences[index]
    return None

In [54]:
from deep_translator import MyMemoryTranslator

def translate_with_mymemory(text, source_lang="ru", target_lang="en"):
    translator = MyMemoryTranslator(source=source_lang, target=target_lang)
    return translator.translate(text)

In [55]:
import time

def double_translate_mymemory(sentence):
    try:
        english = translate_with_mymemory(sentence, "russian", "english")
        time.sleep(1)
        russian_back = translate_with_mymemory(english, "english", "russian")
        time.sleep(1)
        return russian_back
    except Exception as e:
        print(f"Error while processing sentence: {sentence[:50]}")
        print(f"Error: {e}")
        return sentence

In [56]:
with open('dostoevsky_deepseek_ru.txt', 'r', encoding='utf-8') as f:
    dostoevsky_translation = f.read()

In [57]:
original = dostoevsky_sentences[0]
# back_deepseek = double_translate(original, translate_with_deepseek)
# back_google = double_translate(original, translate_with_google)
mymemory_example = double_translate_mymemory(original)

print("Original:", original)
print("\nMyMemory translation:", mymemory_example)

Original: Приступая к описанию недавних и столь странных событий, происшедших в нашем, доселе ничем не отличавшемся городе, я принужден, по неумению моему, начать несколько издалека, а именно некоторыми биографическими подробностями о талантливом и многочтимом Степане Трофимовиче Верховенском.

MyMemory translation: Начав описывать недавние и столь странные события, происходившие в нашем доселе неразличимом городе, я вынужден, по своей неспособности, начать немного издалека, а именно с некоторых биографических подробностей о талантливом и почтенном Степане Трофимовиче Верховенском.


In [58]:
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score
from nltk.tokenize import word_tokenize

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('wordnet')
nltk.download('omw-1.4')

def calculate_metrics(original, back_translated):
    original_tokens = word_tokenize(original.lower())
    back_tokens = word_tokenize(back_translated.lower())
    smoothie = SmoothingFunction().method1
    bleu = sentence_bleu([original_tokens], back_tokens, smoothing_function=smoothie)
    meteor = meteor_score([original_tokens], back_tokens)
    return bleu, meteor

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Varvara\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Varvara\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Varvara\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\Varvara\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [59]:
import pandas as pd

def evaluate_corpus(sentences, translator_name, deepseek_sentences=None):
    results = []
    for i, sent in enumerate(sentences):
        if translator_name == "DeepSeek":
            back_translated = get_deepseek_translation(i, deepseek_sentences)
            if back_translated is None:
                back_translated = sent
        else:
            back_translated = double_translate_mymemory(sent)
        bleu, meteor = calculate_metrics(sent, back_translated)
        results.append({
            "translator": translator_name,
            "bleu": bleu,
            "meteor": meteor
        })
    return results

In [60]:
results = []
results.extend(evaluate_corpus(dostoevsky_sentences[:100], "DeepSeek", dostoevsky_deepseek_sentences))
results.extend(evaluate_corpus(dostoevsky_sentences[:100], "MyMemory"))
results.extend(evaluate_corpus(scientific_sentences[:100], "DeepSeek", scientific_deepseek_sentences))
results.extend(evaluate_corpus(scientific_sentences[:100], "MyMemory"))

df = pd.DataFrame(results)
summary = df.groupby('translator').agg(
    avg_bleu=('bleu', 'mean'),
    avg_meteor=('meteor', 'mean')
).reset_index()
print(summary)

Error while processing sentence: Наконец, сцена опять переменяется, и является дико
Error: Server Error: You made too many requests to the server.According to google, you are allowed to make 5 requests per secondand up to 200k requests per day. You can wait and try again later oryou can try the translate_batch function
Error while processing sentence: Затем вдруг въезжает неописанной красоты юноша на 
Error: Server Error: You made too many requests to the server.According to google, you are allowed to make 5 requests per secondand up to 200k requests per day. You can wait and try again later oryou can try the translate_batch function
Error while processing sentence: Юноша изображает собою смерть, а все народы ее жаж
Error: Server Error: You made too many requests to the server.According to google, you are allowed to make 5 requests per secondand up to 200k requests per day. You can wait and try again later oryou can try the translate_batch function
Error while processing sentence: И, н